In [1]:
from pathlib import Path
import sys
import numpy as np
import glob
from scipy.io import loadmat
from scipy.signal import resample

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

BNCI_RAW_DIR       = PROJECT_ROOT / "datasets" / "bnci_dataset" / "raw"
BNCI_PROCESSED_DIR = PROJECT_ROOT / "datasets" / "bnci_dataset" / "processed"
BNCI_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = BNCI_PROCESSED_DIR / "preprocessed_BNCI_all_subjects.npz"

print("Raw dir    :", BNCI_RAW_DIR)
print("Output path:", OUT_PATH)

Raw dir    : C:\Users\Amrita\Desktop\VS_code\eeg_representation_geometry\datasets\bnci_dataset\raw
Output path: C:\Users\Amrita\Desktop\VS_code\eeg_representation_geometry\datasets\bnci_dataset\processed\preprocessed_BNCI_all_subjects.npz


In [2]:

mat_files = sorted(glob.glob(str(BNCI_RAW_DIR / "*.mat")))

if len(mat_files) == 0:
    raise FileNotFoundError(f"No .mat files found in {BNCI_RAW_DIR}")

print(f"Found {len(mat_files)} .mat files:")
for f in mat_files:
    print("  ", Path(f).name)

Found 18 .mat files:
   A01E.mat
   A01T.mat
   A02E.mat
   A02T.mat
   A03E.mat
   A03T.mat
   A04E.mat
   A04T.mat
   A05E.mat
   A05T.mat
   A06E.mat
   A06T.mat
   A07E.mat
   A07T.mat
   A08E.mat
   A08T.mat
   A09E.mat
   A09T.mat


In [3]:

LABEL_MAP = {"1": 0, "2": 1, "3": 2, "4": 3,
             "left hand": 0, "right hand": 1, "feet": 2, "tongue": 3}

def normalize_label(x):
    if isinstance(x, bytes):
        x = x.decode("utf-8", errors="ignore")
    if isinstance(x, (int, float, np.integer, np.floating)):
        return str(int(x))
    return str(x).strip().lower()

def map_label(x):
    key = normalize_label(x)
    if key in LABEL_MAP:
        return LABEL_MAP[key]
    # Fallback: assign sequentially
    if key not in LABEL_MAP:
        new_id = len(set(LABEL_MAP.values()))
        LABEL_MAP[key] = new_id
    return LABEL_MAP[key]

print("Label map:", LABEL_MAP)

Label map: {'1': 0, '2': 1, '3': 2, '4': 3, 'left hand': 0, 'right hand': 1, 'feet': 2, 'tongue': 3}


In [4]:

TARGET_N_TIMES = 561
TARGET_SFREQ   = 250.0

def extract_epochs_from_elem(elem):

    X       = getattr(elem, "X", None)
    classes = getattr(elem, "y",  None)
    if classes is None:
        classes = getattr(elem, "classes", None)
    fs = float(getattr(elem, "fs", TARGET_SFREQ))

    if X is None or classes is None:
        return None, None

    X       = np.array(X, dtype=np.float32)          
    classes = np.asarray(classes).ravel()

    n_total_samples, n_ch = X.shape
    n_trials               = len(classes)

    if n_trials == 0:
        return None, None

    samples_per_trial = n_total_samples // n_trials
    if samples_per_trial < 10:
        return None, None

    epochs, labels = [], []
    for i in range(n_trials):
        st  = i * samples_per_trial
        ed  = st + samples_per_trial
        if ed > n_total_samples:
            break
        seg = X[st:ed, :].T                          
        epochs.append(seg)
        labels.append(map_label(classes[i]))

    if len(epochs) == 0:
        return None, None

    epochs = np.stack(epochs).astype(np.float32)     
    labels = np.array(labels, dtype=int)

    # Resample to TARGET_N_TIMES if needed
    if epochs.shape[2] != TARGET_N_TIMES:
        resampled = np.zeros((epochs.shape[0], epochs.shape[1], TARGET_N_TIMES), dtype=np.float32)
        for t in range(epochs.shape[0]):
            for ch in range(epochs.shape[1]):
                resampled[t, ch, :] = resample(epochs[t, ch, :], TARGET_N_TIMES)
        epochs = resampled

    return epochs, labels

In [5]:
# ── Parse subject ID from filename ────────────────────────────────────────
import re

def subject_id_from_filename(filepath):
    """
    Extract 0-indexed subject ID from BNCI filename.
    A01T.mat -> 0, A02T.mat -> 1, ..., A09T.mat -> 8
    """
    name = Path(filepath).stem          # e.g. 'A01T'
    match = re.search(r'A(\d+)', name, re.IGNORECASE)
    if match:
        return int(match.group(1)) - 1  # 1-indexed -> 0-indexed
    raise ValueError(f"Cannot parse subject ID from filename: {filepath}")

# Quick sanity check
for f in mat_files[:3]:
    print(f"  {Path(f).name} -> subject_id={subject_id_from_filename(f)}")

  A01E.mat -> subject_id=0
  A01T.mat -> subject_id=0
  A02E.mat -> subject_id=1


In [6]:

all_epochs      = []
all_labels      = []
all_subject_ids = []

subject_trial_counts = {}

for mf in mat_files:
    subj_id = subject_id_from_filename(mf)
    print(f"\nLoading {Path(mf).name}  (subject {subj_id}) ...")

    mat      = loadmat(mf, squeeze_me=True, struct_as_record=False)
    data_arr = mat.get("data", None)

    if data_arr is None:
        print(f"  WARNING: no 'data' key in {Path(mf).name}, skipping")
        continue

    file_epochs = []
    file_labels = []

    
    for elem in np.atleast_1d(data_arr).flat:
        epochs, labels = extract_epochs_from_elem(elem)
        if epochs is None:
            continue
        file_epochs.append(epochs)
        file_labels.append(labels)

    if len(file_epochs) == 0:
        print(f"  WARNING: no valid epochs extracted from {Path(mf).name}")
        continue

    file_X = np.concatenate(file_epochs, axis=0)
    file_y = np.concatenate(file_labels, axis=0)
    file_s = np.full(len(file_y), subj_id, dtype=np.int64)

    print(f"  Extracted {len(file_y)} trials, shape {file_X.shape}")
    print(f"  Label distribution: {dict(zip(*np.unique(file_y, return_counts=True)))}")

    all_epochs.append(file_X)
    all_labels.append(file_y)
    all_subject_ids.append(file_s)

    subject_trial_counts[subj_id] = subject_trial_counts.get(subj_id, 0) + len(file_y)

if len(all_epochs) == 0:
    raise RuntimeError("No epochs extracted from any file. Check BNCI_RAW_DIR.")

X_all = np.concatenate(all_epochs,      axis=0)
y_all = np.concatenate(all_labels,      axis=0)
s_all = np.concatenate(all_subject_ids, axis=0)

print(f"\n{'='*55}")
print(f"TOTAL: {X_all.shape[0]} trials  shape={X_all.shape}")
print(f"Subjects found: {sorted(set(s_all.tolist()))}")
print(f"Per-subject trial counts: {subject_trial_counts}")
print(f"Label distribution: {dict(zip(*np.unique(y_all, return_counts=True)))}")


Loading A01E.mat  (subject 0) ...
  Extracted 288 trials, shape (288, 25, 561)
  Label distribution: {np.int64(0): np.int64(72), np.int64(1): np.int64(72), np.int64(2): np.int64(72), np.int64(3): np.int64(72)}

Loading A01T.mat  (subject 0) ...
  Extracted 288 trials, shape (288, 25, 561)
  Label distribution: {np.int64(0): np.int64(72), np.int64(1): np.int64(72), np.int64(2): np.int64(72), np.int64(3): np.int64(72)}

Loading A02E.mat  (subject 1) ...
  Extracted 288 trials, shape (288, 25, 561)
  Label distribution: {np.int64(0): np.int64(72), np.int64(1): np.int64(72), np.int64(2): np.int64(72), np.int64(3): np.int64(72)}

Loading A02T.mat  (subject 1) ...
  Extracted 288 trials, shape (288, 25, 561)
  Label distribution: {np.int64(0): np.int64(72), np.int64(1): np.int64(72), np.int64(2): np.int64(72), np.int64(3): np.int64(72)}

Loading A03E.mat  (subject 2) ...
  Extracted 288 trials, shape (288, 25, 561)
  Label distribution: {np.int64(0): np.int64(72), np.int64(1): np.int64(72),

In [7]:

print("\n===== SANITY CHECKS =====")

n_subjects = len(np.unique(s_all))
print(f"  Subjects: {n_subjects}  {'OK' if n_subjects == 9 else 'WARNING: expected 9'}")


n_classes = len(np.unique(y_all))
print(f"  Classes:  {n_classes}  {'OK' if n_classes == 4 else 'WARNING: expected 4'}")


print(f"  X shape:  {X_all.shape}  (expected N x 25 x 561)")
assert X_all.shape[1] == 25 and X_all.shape[2] == 561, "Channel/time mismatch"

assert not np.isnan(X_all).any(), "NaNs found in X!"
print(f"  NaN check: OK")


print("\n  Per-subject class distribution:")
for s in sorted(np.unique(s_all)):
    idx  = s_all == s
    dist = dict(zip(*np.unique(y_all[idx], return_counts=True)))
    print(f"    Subject {s}: {X_all[idx].shape[0]} trials  {dist}")


===== SANITY CHECKS =====
  Subjects: 9  OK
  Classes:  4  OK
  X shape:  (5184, 25, 561)  (expected N x 25 x 561)
  NaN check: OK

  Per-subject class distribution:
    Subject 0: 576 trials  {np.int64(0): np.int64(144), np.int64(1): np.int64(144), np.int64(2): np.int64(144), np.int64(3): np.int64(144)}
    Subject 1: 576 trials  {np.int64(0): np.int64(144), np.int64(1): np.int64(144), np.int64(2): np.int64(144), np.int64(3): np.int64(144)}
    Subject 2: 576 trials  {np.int64(0): np.int64(144), np.int64(1): np.int64(144), np.int64(2): np.int64(144), np.int64(3): np.int64(144)}
    Subject 3: 576 trials  {np.int64(0): np.int64(144), np.int64(1): np.int64(144), np.int64(2): np.int64(144), np.int64(3): np.int64(144)}
    Subject 4: 576 trials  {np.int64(0): np.int64(144), np.int64(1): np.int64(144), np.int64(2): np.int64(144), np.int64(3): np.int64(144)}
    Subject 5: 576 trials  {np.int64(0): np.int64(144), np.int64(1): np.int64(144), np.int64(2): np.int64(144), np.int64(3): np.int64

In [8]:

meta = {
    "sfreq"       : TARGET_SFREQ,
    "dataset"     : "BNCI2014-001_all_subjects",
    "n_subjects"  : int(len(np.unique(s_all))),
    "n_channels"  : int(X_all.shape[1]),
    "n_times"     : int(X_all.shape[2]),
    "n_trials"    : int(len(y_all)),
    "label_map"   : LABEL_MAP,
    "evaluation"  : "Leave-One-Subject-Out (LOSO) recommended",
}

np.savez_compressed(
    OUT_PATH,
    X           = X_all.astype(np.float32),
    y           = y_all.astype(np.int64),
    subject_ids = s_all.astype(np.int64),
    meta        = meta
)

print(f"Saved to: {OUT_PATH}")
print(f"File size: {OUT_PATH.stat().st_size / 1e6:.1f} MB")
print(f"\nTotal trials: {len(y_all)}  (was 640, now {len(y_all)})")
print(f"Increase: {len(y_all)/640:.1f}x more data")

Saved to: C:\Users\Amrita\Desktop\VS_code\eeg_representation_geometry\datasets\bnci_dataset\processed\preprocessed_BNCI_all_subjects.npz
File size: 270.3 MB

Total trials: 5184  (was 640, now 5184)
Increase: 8.1x more data
